# Avant le TP5 : configuration d'un client, erreurs de connexion et anatomie d'une ACL

Le TP5 fait travailler trois comptes, chacun avec son propre client Python. Cette démonstration montre, sans toucher à la sécurité du cluster :

1. comment un fichier `.properties` de client Java se traduit en configuration Python ;
2. comment lire une erreur de connexion avec la fonction `connexion()` du TP ;
3. de quoi est faite une ACL, et comment on la crée, la lit et la retire ;
4. où le broker journalise ses décisions d'autorisation.

Sorties de référence : exécution du 07/11/2026 22:41 UTC sur le lab.

In [1]:
import logging
import os

from confluent_kafka import KafkaException
from confluent_kafka.admin import (AclBinding, AclBindingFilter, AclOperation, AclPermissionType, AdminClient,
                                   ResourcePatternType, ResourceType)

BS = os.environ["BS"]
admin = AdminClient({"bootstrap.servers": BS})

## 1. Du fichier Java au dictionnaire Python

Les outils Java lisent leur configuration de sécurité dans un fichier passé par `--command-config`. Voici celui du listener INTERNAL :

In [2]:
!docker exec kafka-tools grep -v '^#' /opt/lab/clients/plaintext.properties

security.protocol=PLAINTEXT


En Python, la même configuration est un dictionnaire. Les noms sont souvent identiques, mais pas toujours : le TP donne un tableau de correspondance pour SASL et TLS.

In [3]:
config_interne = {"bootstrap.servers": BS, "security.protocol": "PLAINTEXT"}
print(config_interne)

{'bootstrap.servers': 'kafka-1:19092,kafka-2:19092,kafka-3:19092', 'security.protocol': 'PLAINTEXT'}


## 2. Lire une erreur de connexion

En cas d'échec, librdkafka réessaie sans fin et écrit une ligne de journal à chaque tentative. La fonction `connexion()` du TP fait un seul essai de 10 s, range ces lignes au lieu de les afficher, et renvoie un résultat lisible : `("OK", topics)` ou `(code d'erreur, premier message)`.

In [4]:
class _Journal(logging.Handler):
    def __init__(self):
        super().__init__(logging.INFO)
        self.lignes = []

    def emit(self, enregistrement):
        self.lignes.append((enregistrement.levelno, enregistrement.getMessage().split("]: ", 1)[-1]))


def connexion(config):
    journal = _Journal()
    logger = logging.getLogger("demo.connexion.%d" % id(journal))
    logger.propagate = False
    logger.setLevel(logging.INFO)
    logger.addHandler(journal)
    erreurs = []
    client = AdminClient({**config, "logger": logger, "error_cb": erreurs.append})
    try:
        return "OK", sorted(client.list_topics(timeout=10).topics)
    except KafkaException as e:
        client.poll(1)
        code = (erreurs[0] if erreurs else e.args[0]).name()
        # Le message le plus grave : une deconnexion est journalisee en INFO, un refus d'authentification en ERROR
        lignes = sorted(journal.lignes, key=lambda ligne: -ligne[0])
        return code, lignes[0][1] if lignes else e.args[0].str()


print(connexion(config_interne))
print(connexion({"bootstrap.servers": "kafka-1:9999"}))

('OK', ['__consumer_offsets', 'loadgen'])


('_TRANSPORT', 'kafka-1:9999/bootstrap: Connect to ipv4#172.22.0.2:9999 failed: Connection refused (after 0ms in state CONNECT)')


Le code d'erreur commence par `_` quand l'erreur est détectée par le client lui-même (`_TRANSPORT`, `_AUTHENTICATION`, `_SSL`) ; sans `_`, c'est une réponse du broker (`TOPIC_AUTHORIZATION_FAILED`).

## 3. Anatomie d'une ACL

Une ACL dit : **qui** (principal), depuis **où** (hôte), peut ou ne peut pas (permission) faire **quoi** (opération) sur **quelle ressource** (type, nom, motif). Exemple sur un compte fictif `demo`, qui n'existe pas : une ACL peut viser un compte avant sa création.

In [5]:
acl = AclBinding(ResourceType.TOPIC, "demo-tp5", ResourcePatternType.LITERAL, "User:demo", "*",
                 AclOperation.READ, AclPermissionType.ALLOW)
admin.create_acls([acl])[acl].result()
print("cree :", acl)

cree : AclBinding(ResourceType.TOPIC,demo-tp5,ResourcePatternType.LITERAL,User:demo,*,AclOperation.READ,AclPermissionType.ALLOW)


In [6]:
!docker exec kafka-tools kafka-acls.sh --bootstrap-server $BS --list --principal User:demo

ACLs for principal `User:demo`
Current ACLs for resource `ResourcePattern(resourceType=TOPIC, name=demo-tp5, patternType=LITERAL)`:
	(principal=User:demo, host=*, operation=READ, permissionType=ALLOW)



Pour lire ou retirer des ACL, on passe un **filtre** : mêmes champs, mais `None` ou `ANY` veulent dire « n'importe lequel ».

In [7]:
filtre = AclBindingFilter(ResourceType.ANY, None, ResourcePatternType.ANY, "User:demo", None,
                          AclOperation.ANY, AclPermissionType.ANY)
print("lues :", admin.describe_acls(filtre).result())
print("retirees :", admin.delete_acls([filtre])[filtre].result())
print("restantes :", admin.describe_acls(filtre).result())

lues : [AclBinding(ResourceType.TOPIC,demo-tp5,ResourcePatternType.LITERAL,User:demo,*,AclOperation.READ,AclPermissionType.ALLOW)]


retirees : [AclBinding(ResourceType.TOPIC,demo-tp5,ResourcePatternType.LITERAL,User:demo,*,AclOperation.READ,AclPermissionType.ALLOW)]
restantes : []


## 4. Le journal des décisions

Chaque broker écrit ses refus d'autorisation dans `kafka-authorizer.log`, à côté de ses autres journaux. Le TP le lit avec une commande de terminal.

In [8]:
%%bash
# Fichiers en cours (sans les fichiers archives par heure, dont le nom finit par un chiffre)
docker exec kafka-1 sh -c 'ls /opt/kafka/logs/ | grep -v "[0-9]$"'

controller.log
kafka-authorizer.log
kafka-request.log
kafkaServer-gc.log
log-cleaner.log
server.log


state-change.log


## La méthode du TP5

Ouvrez `tps/jour2/tp5_securite/enonce.ipynb`.

- Le client d'administration `admin` reste celui du listener INTERNAL. Les comptes `admin`, `alice` et `bob` ont chacun leur configuration, par le listener SECURE.
- Les fonctions fournies `connexion()`, `envoyer()` et `lire()` renvoient un code d'erreur : comparez-le à celui de l'outil Java de la version Markdown.
- Un refus est un résultat attendu dans ce TP : lisez le code, puis cherchez la ligne correspondante dans `kafka-authorizer.log`.